In [1]:
"""
Combined 4-panel figure: ROC curves (A) + top-feature importance bar plots
(B: V4, C: Metabolomics, D: V4+Metabolomics) for High vs Low Sebaceous Skin
classification.

Assumes the following objects already exist in your session (as in your
notebook, after data loading / filtering):
    v4_table            : DataFrame, ASV-level V4 table (samples x features)
    metabolomics_table  : DataFrame, metabolomics table (samples x features)
    combined_table_ASV  : DataFrame, concatenation of v4_table + metabolomics_table
    metadata            : DataFrame with 'Extreme_Sebum' and 'Subject_ID' columns
    taxonomy            : DataFrame with a 'Taxon' column, indexed by ASV FeatureID
                           (this is what gives V4 / combined-panel bars their
                           readable taxonomy labels instead of raw ASV hashes)

Feature labeling logic (panels B, C, D):
    For every top feature, we look it up first in the taxonomy table (ASV ->
    Taxon string) and, failing that, in the metabolomics annotation CSV
    (Feature -> Compound_Name / mz / RT / ClassyFire superclass). This means
    the combined "V4 and Metabolomics" panel correctly labels each bar
    regardless of whether it's a microbial taxon or a metabolite.
"""

import os
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from scipy import interp

In [2]:
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
import pandas as pd
import matplotlib.pyplot as plt
import os
from qiime2.plugins.gemelli.actions import ctf, rpca
from qiime2.plugins.emperor.visualizers import biplot, plot
from qiime2.plugins.feature_table.methods import filter_samples
from gemelli.preprocessing import matrix_rclr
from skbio.stats.composition import clr
from biom import load_table

In [3]:
warnings.filterwarnings("ignore")

plt.style.use("seaborn-whitegrid")
sns.set_context("paper", font_scale=1.5)
sns.set_style("ticks")

# ----------------------------------------------------------------------
# Fixed color mapping for ClassyFire superclasses (metabolomics panels)
# ----------------------------------------------------------------------
SUPERCLASS_COLORS = {
    "Alkaloids and derivatives": "#e41a1c",
    "Benzenoids": "#377eb8",
    "Hydrocarbons": "#4daf4a",
    "Lignans, neolignans and related compounds": "#984ea3",
    "Lipids and lipid-like molecules": "#ff7f00",
    "Nucleosides, nucleotides, and analogues": "#a65628",
    "Organic 1,3-dipolar compounds": "#f781bf",
    "Organic acids and derivatives": "#2ca25e",
    "Organic nitrogen compounds": "#66c2a5",
    "Organic oxygen compounds": "#8da0cb",
    "Organic salts": "#8da0cb",
    "Organoheterocyclic compounds": "#e78ac3",
    "Organosulfur compounds": "#a6d854",
    "Phenylpropanoids and polyketides": "#ffd92f",
    "Unknown": "#bdbdbd",
}
MICROBIAL_TAXON_COLOR = "gray"
MICROBIAL_TAXON_LEGEND_LABEL = "Microbial taxon (16S)"

# Default number of top features to show per table. V4 gets 10; the
# metabolomics-containing panels get 20. Override via `n_top_map` in
# build_combined_figure() if needed.
DEFAULT_N_TOP = {
    "V4": 10,
    "Metabolomics": 20,
    "V4 and Metabolomics": 20,
}


# ----------------------------------------------------------------------
# Cross-validation utilities
# ----------------------------------------------------------------------
def group_stratified_kfold(X, y, groups, n_splits=5, random_state=42):
    """Group-aware stratified k-fold split (keeps Subject_ID groups intact)."""
    unique_groups = np.unique(groups)
    np.random.seed(random_state)
    np.random.shuffle(unique_groups)

    group_label_dist = {
        g: {label: np.sum(y[groups == g] == label) for label in np.unique(y)}
        for g in unique_groups
    }

    folds = [[] for _ in range(n_splits)]
    fold_label_dist = [{label: 0 for label in np.unique(y)} for _ in range(n_splits)]

    sorted_groups = sorted(unique_groups, key=lambda g: sum(groups == g), reverse=True)

    for group in sorted_groups:
        best_fold, min_imbalance = 0, float("inf")
        group_size = sum(groups == group)

        for fold_idx in range(n_splits):
            temp_dist = fold_label_dist[fold_idx].copy()
            for label, count in group_label_dist[group].items():
                temp_dist[label] += count

            fold_size = sum(temp_dist.values())
            proportions = (
                [0] * len(temp_dist) if fold_size == 0
                else [c / fold_size for c in temp_dist.values()]
            )
            imbalance = np.var(proportions) + fold_size / (sum(groups.shape) / n_splits)

            if imbalance < min_imbalance:
                min_imbalance, best_fold = imbalance, fold_idx

        folds[best_fold].extend(np.where(groups == group)[0])
        for label, count in group_label_dist[group].items():
            fold_label_dist[best_fold][label] += count

    train_test_indices = []
    for i in range(n_splits):
        test_idx = np.array(folds[i])
        train_idx = np.concatenate([folds[j] for j in range(n_splits) if j != i])
        train_test_indices.append((train_idx, test_idx))

    return train_test_indices


def run_group_stratified_cv(X, y, groups, n_splits=5):
    """Run RF classification with group-stratified CV; return per-fold results
    and mean/std feature importances."""
    folds = group_stratified_kfold(X, y, groups, n_splits=n_splits)

    cv_results = []
    feature_importances = pd.DataFrame(index=X.columns)

    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"Skipping fold {i + 1}: insufficient class representation")
            continue

        clf = RandomForestClassifier(n_estimators=1000, random_state=42)
        clf.fit(X_train, y_train)
        probas = clf.predict_proba(X_test)

        feature_importances[f"fold_{i}"] = clf.feature_importances_

        fpr, tpr, _ = roc_curve(y_test, probas[:, 1])
        roc_auc = auc(fpr, tpr)

        cv_results.append({"fpr": fpr, "tpr": tpr, "auc": roc_auc, "fold": i})

    feature_importances["mean_importance"] = feature_importances.mean(axis=1)
    feature_importances["std_importance"] = feature_importances.std(axis=1)
    feature_importances = feature_importances.sort_values("mean_importance", ascending=False)

    return cv_results, feature_importances


def compute_roc_summary(cv_results):
    """Interpolate fold ROC curves onto a common FPR grid; return mean/std TPR + AUC."""
    mean_fpr = np.linspace(0, 1, 100)
    tprs, aucs = [], []

    for result in cv_results:
        tpr_interp = interp(mean_fpr, result["fpr"], result["tpr"])
        tpr_interp[0] = 0.0
        tprs.append(tpr_interp)
        aucs.append(result["auc"])

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)
    mean_auc, std_auc = np.mean(aucs), np.std(aucs)

    return mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc, aucs


def compute_pairwise_comparisons(fold_aucs_dict):
    """Wilcoxon / paired t-test comparisons between methods, per task."""
    from scipy.stats import wilcoxon, ttest_rel

    results = []
    for task, methods_dict in fold_aucs_dict.items():
        methods = list(methods_dict.keys())
        for method1, method2 in combinations(methods, 2):
            aucs1, aucs2 = methods_dict[method1], methods_dict[method2]
            min_len = min(len(aucs1), len(aucs2))
            if min_len < 2:
                continue
            aucs1, aucs2 = aucs1[:min_len], aucs2[:min_len]

            try:
                _, p_wilcoxon = wilcoxon(aucs1, aucs2)
            except Exception:
                p_wilcoxon = np.nan
            _, p_ttest = ttest_rel(aucs1, aucs2)

            results.append({
                "Task": task, "Method 1": method1, "Method 2": method2,
                "Mean AUC 1": np.mean(aucs1), "Mean AUC 2": np.mean(aucs2),
                "AUC Difference": np.mean(aucs1) - np.mean(aucs2),
                "p-value (Wilcoxon)": p_wilcoxon, "p-value (t-test)": p_ttest,
                "Significant (p<0.05)": (p_wilcoxon < 0.05) if not np.isnan(p_wilcoxon) else (p_ttest < 0.05),
            })

    return pd.DataFrame(results)


# ----------------------------------------------------------------------
# Taxonomy helper
# ----------------------------------------------------------------------
def clean_taxonomy(tax_str):
    """Collapse a full QIIME taxonomy string down to the last two informative
    levels, dropping 'uncultured' entries (mirrors your original cell 56 logic)."""
    parts = [p.strip() for p in str(tax_str).split(";") if p.strip()]
    parts = [p for p in parts if "uncultured" not in p.lower()]
    if len(parts) >= 2:
        return "; ".join(parts[-2:])
    elif parts:
        return parts[-1]
    return str(tax_str)


def build_taxonomy_map(taxonomy_df):
    """Build {FeatureID(str): Taxon string} from a taxonomy DataFrame indexed
    by ASV FeatureID with a 'Taxon' column."""
    if taxonomy_df is None:
        return {}
    return {str(k): v for k, v in taxonomy_df["Taxon"].to_dict().items()}


# ----------------------------------------------------------------------
# Feature-label formatting helper (shared by panels B, C, D)
# ----------------------------------------------------------------------
def label_feature(feat, taxonomy_map, annotation_df):
    """
    Determine the display label / bar color / legend category for a single
    feature, regardless of which panel it appears in. Checks taxonomy first
    (microbial ASV), then the metabolomics annotation table (metabolite).
    Falls back to a generic 'Feature <id>' label if neither matches -- this
    is what lets the combined V4+Metabolomics panel label each bar correctly.
    """
    feat_str = str(feat)

    # 1) Microbial ASV -> taxonomy label
    if feat_str in taxonomy_map:
        label = clean_taxonomy(taxonomy_map[feat_str])
        return label, MICROBIAL_TAXON_COLOR, MICROBIAL_TAXON_LEGEND_LABEL

    # 2) Metabolite -> annotation label
    if annotation_df is not None:
        row = annotation_df[annotation_df["Feature"].astype(str) == feat_str]
        if not row.empty:
            name = row["Compound_Name"].values[0]
            mz = round(row["mz"].values[0], 3)
            rt = round(row["RT"].values[0], 1)
            label = (
                f"mz:{mz}, RT:{rt} [Feature {feat_str}]"
                if pd.isna(name) or name == ""
                else f"{name} (mz:{mz}, RT:{rt}) [Feature {feat_str}]"
            )
            superclass = (
                row["ClassyFire#superclass"].values[0]
                if "ClassyFire#superclass" in row.columns
                else "Unknown"
            )
            superclass = superclass if pd.notna(superclass) and superclass != "" else "Unknown"
            color = SUPERCLASS_COLORS.get(superclass, "#bdbdbd")
            return label, color, superclass

    # 3) Unmatched fallback
    return f"Feature {feat_str}", "#bdbdbd", "Unknown"


def format_top_features(features_df, taxonomy_map, annotation_df=None, n_top=10):
    """Return top-N features with display labels + bar colors + legend
    categories used, for a given feature-importance table."""
    top_df = features_df.sort_values("mean_importance", ascending=False).head(n_top).copy()
    top_df = top_df.reset_index().rename(columns={"index": "feature"})

    labels, colors, legend_categories = [], [], []
    for feat in top_df["feature"]:
        label, color, legend_cat = label_feature(feat, taxonomy_map, annotation_df)
        labels.append(label)
        colors.append(color)
        legend_categories.append(legend_cat)

    top_df["label"] = labels
    top_df["color"] = colors
    top_df = top_df.sort_values("mean_importance", ascending=True)  # ascending for barh
    return top_df, legend_categories


def draw_feature_panel(ax, top_df, data_type, legend_categories, show_legend):
    """Draw a horizontal bar chart of top features on a given axis."""
    n_bars = len(top_df)
    ax.barh(
        y=top_df["label"],
        width=top_df["mean_importance"],
        xerr=top_df["std_importance"],
        color=top_df["color"],
        edgecolor="black",
    )
    ax.set_title(f"Top Features - {data_type} (High vs Low Sebaceous Skin)", fontsize=13)
    ax.set_xlabel("Mean Importance")
    ax.invert_yaxis()
    ax.grid(True, axis="x", linestyle="--", alpha=0.6)
    ax.tick_params(axis="y", labelsize=9 if n_bars <= 10 else 7.5)

    if show_legend and legend_categories:
        used_unique = sorted(set(legend_categories))
        legend_patches = []
        for cat in used_unique:
            if cat == MICROBIAL_TAXON_LEGEND_LABEL:
                legend_patches.append(Patch(facecolor=MICROBIAL_TAXON_COLOR, edgecolor="black", label=cat))
            else:
                legend_patches.append(Patch(facecolor=SUPERCLASS_COLORS.get(cat, "#bdbdbd"), edgecolor="black", label=cat))
        ax.legend(
            handles=legend_patches, title="Feature Category",
            loc="upper center", bbox_to_anchor=(0.5, -0.15),
            ncol=2, fontsize=8, title_fontsize=9, frameon=False,
        )


def add_panel_label(ax, letter):
    """Bold capital panel label (A, B, C, D) in the top-left corner of a panel."""
    ax.text(
        -0.12, 1.08, letter, transform=ax.transAxes,
        fontsize=20, fontweight="bold", va="top", ha="left",
    )


# ----------------------------------------------------------------------
# Main entry point: build the full 2x2 combined figure
# ----------------------------------------------------------------------
def build_combined_figure(
    tables,
    metadata,
    label1="High",
    label2="Low",
    n_splits=5,
    annotation_path=None,
    taxonomy=None,
    n_top_map=None,
    save_dir="../output/ML",
    filename="roc_and_top_features_HvsL_combined",
):
    """
    Parameters
    ----------
    tables : dict
        e.g. {'V4': v4_table, 'Metabolomics': metabolomics_table,
              'V4 and Metabolomics': combined_table_ASV}
        All tables are overlaid together in panel A (ROC); 'V4' -> panel B,
        'Metabolomics' -> panel C, 'V4 and Metabolomics' -> panel D.
    metadata : DataFrame
        Must contain 'Extreme_Sebum' and 'Subject_ID' columns.
    annotation_path : str or None
        Path to metabolite annotation CSV (Feature, Compound_Name, mz, RT,
        'ClassyFire#superclass'). Required for readable metabolomics labels.
    taxonomy : DataFrame or None
        Your `taxonomy` table (index = ASV FeatureID, column 'Taxon').
        Required for readable microbial taxon labels in panels B and D.
    n_top_map : dict or None
        Number of top features to plot per table, e.g.
        {'V4': 10, 'Metabolomics': 20, 'V4 and Metabolomics': 20}.
        Falls back to DEFAULT_N_TOP for any table not specified.
    save_dir : str
        Output directory for the PNG/SVG.
    filename : str
        Base filename (without extension).
    """
    os.makedirs(save_dir, exist_ok=True)

    annotation_df = None
    if annotation_path:
        annotation_df = pd.read_csv(os.path.expanduser(annotation_path))

    taxonomy_map = build_taxonomy_map(taxonomy)

    n_top_map = {**DEFAULT_N_TOP, **(n_top_map or {})}

    # ---- Run CV for each table once, reuse for panel A + B/C/D ----
    roc_summaries = {}        # table_name -> (mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc)
    feature_importances = {}  # table_name -> full importances DataFrame
    fold_aucs = {}

    meta_subset = metadata[metadata["Extreme_Sebum"].isin([label1, label2])]

    for table_name, table in tables.items():
        common_samples = table.index.intersection(meta_subset.index)
        X = table.loc[common_samples]
        meta_filtered = meta_subset.loc[common_samples]

        if len(common_samples) < 10:
            print(f"Skipping {table_name}: insufficient samples ({len(common_samples)})")
            continue

        y = meta_filtered["Extreme_Sebum"].map({label1: 0, label2: 1})
        groups = meta_filtered["Subject_ID"]

        cv_results, feat_imp = run_group_stratified_cv(X, y, groups, n_splits=n_splits)
        if len(cv_results) < 2:
            print(f"Skipping {table_name}: insufficient CV results")
            continue

        mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc, aucs = compute_roc_summary(cv_results)
        roc_summaries[table_name] = (mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc)
        feature_importances[table_name] = feat_imp
        fold_aucs[table_name] = aucs

    pairwise_stats = compute_pairwise_comparisons({f"{label1}_vs_{label2}": fold_aucs})

    # ---- Build combined figure (2x2 grid) ----
    fig = plt.figure(figsize=(22, 16))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.6, wspace=0.35)

    ax_roc = fig.add_subplot(gs[0, 0])       # A: top-left
    ax_v4 = fig.add_subplot(gs[1, 0])        # B: bottom-left
    ax_metab = fig.add_subplot(gs[0, 1])     # C: top-right
    ax_combo = fig.add_subplot(gs[1, 1])     # D: bottom-right

    # --- Panel A: ROC curves, all tables overlaid ---
    for table_name, (mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc) in roc_summaries.items():
        line, = ax_roc.plot(mean_fpr, mean_tpr, lw=2,
                             label=f"{table_name} (AUC = {mean_auc:.2f} \u00b1 {std_auc:.2f})")
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax_roc.fill_between(mean_fpr, tprs_lower, tprs_upper, alpha=0.3, color=line.get_color())

    ax_roc.plot([0, 1], [0, 1], "k--", lw=1)
    ax_roc.set_title(f"Classification Performance on {label1} vs {label2} Sebaceous Skin", fontsize=15)
    ax_roc.set_xlabel("False Positive Rate", fontsize=13)
    ax_roc.set_ylabel("True Positive Rate", fontsize=13)
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.grid(True, linestyle="--", alpha=0.7)
    ax_roc.legend(loc="lower right", fontsize=9)

    # --- Panels B, C, D: top-N feature importance bar charts ---
    panel_axes = {"V4": ax_v4, "Metabolomics": ax_metab, "V4 and Metabolomics": ax_combo}

    for table_name, ax in panel_axes.items():
        if table_name not in feature_importances:
            ax.axis("off")
            continue
        n_top = n_top_map.get(table_name, 10)
        top_df, legend_categories = format_top_features(
            feature_importances[table_name], taxonomy_map, annotation_df, n_top=n_top
        )
        # Show legend whenever the panel contains metabolite and/or taxon
        # features worth distinguishing (i.e. not the plain V4-only panel).
        show_legend = table_name in ("Metabolomics", "V4 and Metabolomics")
        draw_feature_panel(ax, top_df, table_name, legend_categories, show_legend)

    # --- Panel labels ---
    add_panel_label(ax_roc, "A")
    add_panel_label(ax_v4, "B")
    add_panel_label(ax_metab, "C")
    add_panel_label(ax_combo, "D")

    fig.tight_layout()

    # ---- Save ----
    png_path = os.path.join(save_dir, f"{filename}.png")
    svg_path = os.path.join(save_dir, f"{filename}.svg")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    print(f"Saved:\n  {png_path}\n  {svg_path}")

    return fig, feature_importances, pairwise_stats



In [4]:
v4_table_original = Artifact.load('../data/217045_rarefied_table_RefHit_filtered.qza').view(pd.DataFrame)
#already filtered out middle samples
metabolomics_table = Artifact.load('../data/Metabolomics_data_sample_extreme_sebum_05012025.qza').view(pd.DataFrame)
#full table
#metabolomics_table = Artifact.load('../data/Metabolomics_data_sample_05022025.qza').view(pd.DataFrame)

metadata = pd.read_csv('../data/Shiseido_metadata_with_alpha_RefHit.tsv', sep='\t', index_col=0)
taxonomy = pd.read_csv('../data/217041_taxonomy_RefHit.tsv', sep='\t', index_col=0)

In [5]:
print(v4_table_original)

                      TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGTGCAGGCGGCTCGATAAGTCTGATGTGAAAGCCTTCGGCTCAACCGGAGAATTGCATCAGAAACTGTCGAGCTTGAGTACAGAAGAGGAGAGTGGAACTCCATG  \
130729.11369.m02.10A                                               16.0                                                                                                        
130729.11369.m02.02B                                                0.0                                                                                                        
130729.11369.m01.10A                                                0.0                                                                                                        
130729.11369.m01.02B                                                0.0                                                                                                        
130729.11369.m02.10B                                                2.0                                                 

In [6]:
#Perform Prevalence filtering before running the RF
# Calculate prevalence of each feature (i.e., proportion of samples where the feature is non-zero)
prevalence = (v4_table_original > 0).sum(axis=0) / v4_table_original.shape[0]

# Filter to keep only features present in at least 10% of samples
filtered_v4_table_original = v4_table_original.loc[:, prevalence >= 0.10]


In [7]:
print(filtered_v4_table_original)

                      TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGTGCAGGCGGCTCGATAAGTCTGATGTGAAAGCCTTCGGCTCAACCGGAGAATTGCATCAGAAACTGTCGAGCTTGAGTACAGAAGAGGAGAGTGGAACTCCATG  \
130729.11369.m02.10A                                               16.0                                                                                                        
130729.11369.m02.02B                                                0.0                                                                                                        
130729.11369.m01.10A                                                0.0                                                                                                        
130729.11369.m01.02B                                                0.0                                                                                                        
130729.11369.m02.10B                                                2.0                                                 

In [8]:
v4_table_original_copy = v4_table_original

In [9]:
v4_table_original = filtered_v4_table_original

In [10]:
#Perform Prevalence filtering before running the RF
# Calculate prevalence of each feature (i.e., proportion of samples where the feature is non-zero)
prevalence = (metabolomics_table > 0).sum(axis=0) / metabolomics_table.shape[0]

# Filter to keep only features present in at least 10% of samples
filtered_metabolomics_table = metabolomics_table.loc[:, prevalence >= 0.10]


In [11]:
metabolomics_table_copy = metabolomics_table

In [12]:
print(metabolomics_table_copy)

                      392  2429         605     1535       2430  1887  \
130729.11369.m01.13A  0.0   0.0   53.770615   0.0000   0.000000   0.0   
130729.11369.m01.14A  0.0   0.0    0.000000   0.0000  56.262432   0.0   
130729.11369.m01.15A  0.0   0.0    0.000000   0.0000  49.949375   0.0   
130729.11369.m01.13B  0.0   0.0  117.737320   0.0000   0.000000   0.0   
130729.11369.m01.14B  0.0   0.0    0.000000   0.0000   0.000000   0.0   
...                   ...   ...         ...      ...        ...   ...   
130729.11369.p58.2B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p58.3B   0.0   0.0    0.000000  52.2255   0.000000   0.0   
130729.11369.p59.1B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p59.2B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p59.3B   0.0   0.0    0.000000   0.0000   0.000000   0.0   

                           1976           45           22      1589  ...  \
130729.11369.m01.13A     0.0000    66.716900   

In [13]:
metabolomics_table = filtered_metabolomics_table

In [14]:
print(metabolomics_table)

                      392  2429         605     1535       2430  1887  \
130729.11369.m01.13A  0.0   0.0   53.770615   0.0000   0.000000   0.0   
130729.11369.m01.14A  0.0   0.0    0.000000   0.0000  56.262432   0.0   
130729.11369.m01.15A  0.0   0.0    0.000000   0.0000  49.949375   0.0   
130729.11369.m01.13B  0.0   0.0  117.737320   0.0000   0.000000   0.0   
130729.11369.m01.14B  0.0   0.0    0.000000   0.0000   0.000000   0.0   
...                   ...   ...         ...      ...        ...   ...   
130729.11369.p58.2B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p58.3B   0.0   0.0    0.000000  52.2255   0.000000   0.0   
130729.11369.p59.1B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p59.2B   0.0   0.0    0.000000   0.0000   0.000000   0.0   
130729.11369.p59.3B   0.0   0.0    0.000000   0.0000   0.000000   0.0   

                           1976           45           22      1589  ...  \
130729.11369.m01.13A     0.0000    66.716900   

In [15]:

# Drop rows with missing values in 'sebum' or 'SubjectID'
clean_metadata = metadata.dropna(subset=['sebum', 'Subject_ID']).copy()



In [16]:
print(clean_metadata)

                         SampleID_2 Sample_name_old  Line_No. PlateNo  \
#SampleID                                                               
130729.11369.m01.01A  11369.m01.01A         m01-01A         1     m01   
130729.11369.m01.01B  11369.m01.01B         m01-01B        13     m01   
130729.11369.m01.02A  11369.m01.02A         m01-02A         2     m01   
130729.11369.m01.02B  11369.m01.02B         m01-02B        14     m01   
130729.11369.m01.03A  11369.m01.03A         m01-03A         3     m01   
...                             ...             ...       ...     ...   
130729.11369.p60.1B    11369.p60.1B          p60-1B       478     p06   
130729.11369.p60.2A    11369.p60.2A          p60-2A       476     p06   
130729.11369.p60.2B    11369.p60.2B          p60-2B       479     p06   
130729.11369.p60.3A    11369.p60.3A          p60-3A       477     p06   
130729.11369.p60.3B    11369.p60.3B          p60-3B       480     p06   

                     WellPosition SmplSite  Subjec

In [17]:
extreme_metadata = clean_metadata[(clean_metadata['sebum'] < 3.5) | (clean_metadata['sebum'] > 16.9)]

# Extract the sample IDs to use for filtering
extreme_sample_ids = extreme_metadata.index.tolist()


In [18]:
print(extreme_sample_ids)

['130729.11369.m01.13A', '130729.11369.m01.13B', '130729.11369.m01.14A', '130729.11369.m01.14B', '130729.11369.m01.15A', '130729.11369.m01.15B', '130729.11369.m02.01A', '130729.11369.m02.01B', '130729.11369.m02.02A', '130729.11369.m02.02B', '130729.11369.m02.04A', '130729.11369.m02.04B', '130729.11369.m02.08A', '130729.11369.m02.08B', '130729.11369.m02.15A', '130729.11369.m02.15B', '130729.11369.m03.02A', '130729.11369.m03.02B', '130729.11369.m03.06A', '130729.11369.m03.06B', '130729.11369.m03.07A', '130729.11369.m03.07B', '130729.11369.m03.08A', '130729.11369.m03.08B', '130729.11369.m03.09A', '130729.11369.m03.09B', '130729.11369.m03.10A', '130729.11369.m03.10B', '130729.11369.m03.11A', '130729.11369.m03.11B', '130729.11369.m03.12A', '130729.11369.m03.12B', '130729.11369.m03.13A', '130729.11369.m03.13B', '130729.11369.m03.15A', '130729.11369.m03.15B', '130729.11369.m04.01A', '130729.11369.m04.01B', '130729.11369.m04.02A', '130729.11369.m04.02B', '130729.11369.m04.08A', '130729.11369.m

In [19]:
# Keep only the samples in extreme_sample_ids
v4_table_extreme = v4_table_original.loc[v4_table_original.index.isin(extreme_sample_ids), :]


In [20]:
# or just keep all samples
v4_table_extreme = v4_table_original

In [21]:
clean_metadata.loc[:, 'Extreme_Sebum'] = clean_metadata.apply(
    lambda row: 'Low' if row.name in extreme_sample_ids and row['sebum'] < 3.5
    else 'High' if row.name in extreme_sample_ids and row['sebum'] > 16.9
    else None,
    axis=1
)


In [22]:
clean_metadata['sebum_group'] = clean_metadata.apply(
    lambda row: (
        'Low' if row['sebum'] < 3.5 else
        'High' if row['sebum'] > 16.9 else
        'Medium'
    ),
    axis=1
)


In [23]:
clean_metadata['sebum_group'].value_counts()


Medium    242
High      120
Low       118
Name: sebum_group, dtype: int64

In [24]:
# Create a shared index between all tables
shared_samples = set(clean_metadata.index).intersection(
    set(v4_table_extreme.index)

)
print(f"\nNumber of samples common to all tables: {len(shared_samples)}")

# Filter all tables to only include samples present in all datasets
metadata = clean_metadata.loc[list(shared_samples)]
v4_table = v4_table_extreme.loc[list(shared_samples)]

# Verify the filtered tables
print("\nAfter filtering to shared samples:")
print(f"Metadata shape: {metadata.shape}")
print(f"V4 table shape: {v4_table.shape}")


Number of samples common to all tables: 468

After filtering to shared samples:
Metadata shape: (468, 101)
V4 table shape: (468, 237)


In [25]:
# Create a shared index between all tables
shared_samples = set(clean_metadata.index).intersection(
    set(v4_table_extreme.index),
    set(metabolomics_table.index)

)
print(f"\nNumber of samples common to all tables: {len(shared_samples)}")

# Filter all tables to only include samples present in all datasets
metadata = clean_metadata.loc[list(shared_samples)]
v4_table = v4_table_extreme.loc[list(shared_samples)]
metabolomics_table = metabolomics_table.loc[list(shared_samples)]

# Verify the filtered tables
print("\nAfter filtering to shared samples:")
print(f"Metadata shape: {metadata.shape}")
print(f"V4 table shape: {v4_table.shape}")
print(f"Metabolomics table shape: {metabolomics_table.shape}")


Number of samples common to all tables: 231

After filtering to shared samples:
Metadata shape: (231, 101)
V4 table shape: (231, 237)
Metabolomics table shape: (231, 518)


In [26]:
assert all(v4_table.index == metabolomics_table.index), "SampleID mismatch!"


In [27]:
extreme_sebum_counts = metadata['Extreme_Sebum'].value_counts(dropna=True)
print(extreme_sebum_counts)


High    119
Low     112
Name: Extreme_Sebum, dtype: int64


In [28]:
# Ensure the index (SampleID) is aligned and no duplicates
combined_table_ASV = pd.concat([v4_table, metabolomics_table], axis=1)
print(f"Processed v4_table and metabolomics table shape: {combined_table_ASV.shape}")

Processed v4_table and metabolomics table shape: (231, 755)


In [29]:
metadata['Extreme_Sebum'].value_counts()
metadata.groupby('Extreme_Sebum')['Subject_ID'].nunique()


Extreme_Sebum
High    32
Low     27
Name: Subject_ID, dtype: int64

In [32]:


 tables = {
     "V4": v4_table,
     "Metabolomics": metabolomics_table,
     "V4 and Metabolomics": combined_table_ASV,
 }

 fig, feature_importances, pairwise_stats = build_combined_figure(
     tables,
     metadata,
     label1="High",
     label2="Low",
     n_splits=5,
     annotation_path="../data/info_feature_complete.csv",
     taxonomy=taxonomy,                     # <-- your taxonomy DataFrame (index=ASV ID, col='Taxon')
     n_top_map={"V4": 10, "Metabolomics": 20, "V4 and Metabolomics": 20},
     save_dir="../output/ML",
     filename="roc_and_top_features_HvsL_combined",
 )

 print(pairwise_stats.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Saved:
  ../output/ML/roc_and_top_features_HvsL_combined.png
  ../output/ML/roc_and_top_features_HvsL_combined.svg
       Task     Method 1            Method 2  Mean AUC 1  Mean AUC 2  AUC Difference  p-value (Wilcoxon)  p-value (t-test)  Significant (p<0.05)
High_vs_Low           V4        Metabolomics      0.6983      0.7962         -0.0978              0.3125            0.2739                 False
High_vs_Low           V4 V4 and Metabolomics      0.6983      0.8068         -0.1085              0.1875            0.2393                 False
High_vs_Low Metabolomics V4 and Metabolomics      0.7962      0.8068         -0.0106              0.1875            0.1996                 False
